# Feature and subset analysis with deficiency, surplus, and miscoding

This notebook illustrates the current `Miscoding` class.

The class exposes two levels of diagnostics:

1. **Feature-level diagnostics** for each individual feature \(X_j\).
2. **Subset-level diagnostics** for a selected set of features \(X_S\).

For a single feature \(X_j\) and target \(Y\), the public metrics are

\[
\delta_j = \frac{K(Y \mid X_j)}{K(Y)},
\qquad
\omega_j = \frac{K(X_j \mid Y)}{K(X_j)},
\qquad
\mu_j = \max\{\delta_j,\omega_j\}.
\]

For a feature subset \(S\), the current implementation uses a redundancy-discounted aggregation:

\[
\delta_S
=
\prod_{j\in S}\delta_j^{\alpha_j},
\qquad
\alpha_j=
\frac{1}{1+\sum_{k\in S,\;k\neq j}\rho_{jk}}.
\]

Subset surplus is computed as a redundancy-weighted average of the individual surpluses, and subset miscoding is

\[
\mu_S=\max\{\delta_S,\omega_S\}.
\]

Feature selection uses a greedy redundancy-penalized procedure: at each step, it adds the feature that most reduces the current subset miscoding.


## 1. Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif
from sklearn.datasets import make_classification

from nescience.miscoding import Miscoding

plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 2. Helper functions


In [ ]:
def show_feature_analysis(metric):
    """Return the standard public feature-level analysis table."""
    return metric.feature_analysis()[[
        "feature_name",
        "is_numeric",
        "code_length",
        "deficiency",
        "surplus",
        "miscoding",
    ]]


def plot_deficiency_surplus(analysis, title):
    """Plot feature-level deficiency and surplus."""
    plot_data = analysis.set_index("feature_name")[[
        "deficiency",
        "surplus",
    ]]
    plot_data.plot(kind="bar")
    plt.ylabel("Value")
    plt.title(title)
    plt.xticks(rotation=45)
    plt.ylim(0, 1.05)
    plt.show()


def plot_miscoding(analysis, title):
    """Plot feature-level miscoding."""
    plot_data = analysis.sort_values("feature_index")
    plt.figure(figsize=(10, 4))
    plt.bar(plot_data["feature_name"], plot_data["miscoding"])
    plt.ylabel("Miscoding")
    plt.title(title)
    plt.xticks(rotation=45)
    plt.ylim(0, 1.05)
    plt.show()


def feature_indices(metric, names):
    """Convert feature names into feature indices."""
    name_to_index = {
        str(name): index
        for index, name in enumerate(metric.feature_names_in_)
    }
    return [name_to_index[name] for name in names]


def subset_summary(metric, names):
    """Return a compact subset-level diagnostic row for named features."""
    indices = feature_indices(metric, names)
    values = metric.subset_analysis(indices)

    return {
        "features": tuple(names),
        "deficiency": values["deficiency"],
        "surplus": values["surplus"],
        "miscoding": values["miscoding"],
        "redundancy_weights": tuple(np.round(values["redundancy_weights"], 4)),
        "feature_weights": tuple(np.round(values["feature_weights"], 2)),
    }


def subset_table(metric, subsets):
    """Build a DataFrame with subset diagnostics for several named subsets."""
    return pd.DataFrame([
        subset_summary(metric, names)
        for names in subsets
    ])


def plot_redundancy(metric, title):
    """Plot the pairwise feature redundancy matrix."""
    redundancy = metric.feature_redundancy()
    plt.figure(figsize=(7, 6))
    plt.imshow(redundancy.values, aspect="auto")
    plt.colorbar(label="Redundancy")
    plt.xticks(range(len(redundancy.columns)), redundancy.columns, rotation=90)
    plt.yticks(range(len(redundancy.index)), redundancy.index)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return redundancy


## 3. A canonical categorical example

We construct a target composed of two independent pieces of information \(A\) and \(B\).

The feature set contains:

- `perfect`: contains exactly the same information as the target;
- `deficient`: contains only part of the target;
- `surplus`: contains the target plus irrelevant information;
- `irrelevant`: contains information unrelated to the target.


In [ ]:
rng = np.random.default_rng(42)
n = 6000

A = rng.integers(0, 4, size=n)
B = rng.integers(0, 4, size=n)
C = rng.integers(0, 4, size=n)
D = rng.integers(0, 16, size=n)

Y = np.array([f"{a}_{b}" for a, b in zip(A, B)])

X = pd.DataFrame({
    "perfect": np.array([f"{a}_{b}" for a, b in zip(A, B)]),
    "deficient": A.astype(str),
    "surplus": np.array([f"{a}_{b}_{c}" for a, b, c in zip(A, B, C)]),
    "irrelevant": D.astype(str),
})

metric_cat = Miscoding(
    X_type="categorical",
    y_type="categorical",
)
metric_cat.fit(X, Y)

analysis_cat = metric_cat.feature_analysis()
show_feature_analysis(metric_cat)


In [ ]:
plot_deficiency_surplus(
    analysis_cat,
    "Categorical example: deficiency and surplus",
)

plot_miscoding(
    analysis_cat,
    "Categorical example: miscoding",
)


Expected interpretation:

- `perfect` should have low deficiency and low surplus.
- `deficient` should have high deficiency.
- `surplus` should have high surplus.
- `irrelevant` should usually have high deficiency and high miscoding.


## 4. A canonical numeric example


In [ ]:
rng = np.random.default_rng(0)

n_signal = 4000
n_zero = 2000
n_total = n_signal + n_zero

signal = rng.normal(size=n_signal)
y = np.concatenate([signal, np.zeros(n_zero)])

X_num = pd.DataFrame({
    "perfect": y + rng.normal(0.0, 0.01, size=n_total),
    "deficient": np.concatenate([signal[:2000], np.zeros(n_total - 2000)]),
    "surplus": np.concatenate([signal, rng.normal(5, size=n_zero)]),
    "irrelevant": rng.normal(size=n_total),
})

metric_num = Miscoding(
    X_type="numeric",
    y_type="numeric",
    n_bins=20,
)
metric_num.fit(X_num, y)

analysis_num = metric_num.feature_analysis()
show_feature_analysis(metric_num)


In [ ]:
plot_deficiency_surplus(
    analysis_num,
    "Numeric example: deficiency and surplus",
)

plot_miscoding(
    analysis_num,
    "Numeric example: miscoding",
)


## 5. Accessing the public arrays directly

The class exposes the three feature-level metrics through dedicated methods.


In [ ]:
metric_table = pd.DataFrame({
    "feature_name": X_num.columns,
    "deficiency": metric_num.feature_deficiency(),
    "surplus": metric_num.feature_surplus(),
    "miscoding": metric_num.feature_miscoding(),
})

metric_table


## 6. Pairwise redundancy between features

The current implementation computes pairwise redundancy between features during `fit`.

The redundancy matrix is useful for understanding why duplicated features should not be counted multiple times when computing subset-level miscoding.


In [ ]:
X_redundant = X_num.copy()
X_redundant["perfect_copy"] = X_redundant["perfect"] + rng.normal(0.0, 0.01, size=n_total)

metric_redundant = Miscoding(
    X_type="numeric",
    y_type="numeric",
    n_bins=20,
)
metric_redundant.fit(X_redundant, y)

show_feature_analysis(metric_redundant)


In [ ]:
redundancy_redundant = plot_redundancy(
    metric_redundant,
    "Numeric example: pairwise feature redundancy",
)

redundancy_redundant.round(3)


## 7. Subset-level miscoding

`miscoding_subset(...)` and `subset_analysis(...)` use the redundancy-discounted product of deficiencies.

This means that adding a redundant copy of an already selected feature should not greatly improve the subset score.


In [ ]:
subsets = [
    ["perfect"],
    ["perfect", "perfect_copy"],
    ["perfect", "surplus"],
    ["deficient"],
    ["deficient", "surplus"],
    ["irrelevant"],
    ["perfect", "irrelevant"],
]

subset_table(metric_redundant, subsets)


In [ ]:
comparison_rows = []

for names in subsets:
    indices = feature_indices(metric_redundant, names)
    comparison_rows.append({
        "features": tuple(names),
        "deficiency": metric_redundant.miscoding_subset(indices, mode="deficiency"),
        "surplus": metric_redundant.miscoding_subset(indices, mode="surplus"),
        "miscoding": metric_redundant.miscoding_subset(indices, mode="miscoding"),
    })

pd.DataFrame(comparison_rows)


## 8. Greedy redundancy-penalized feature selection

`select_features()` now selects features by greedy minimization of subset-level miscoding.

At each step, the method tests every unselected feature and adds the one that produces the largest reduction in the current subset miscoding.


In [ ]:
selection = metric_redundant.select_features(return_details=True)

selection["selected_feature_names"]


In [ ]:
selection["path"]


In [ ]:
selection["subset"]


In [ ]:
selection["features"][[
    "feature_name",
    "code_length",
    "deficiency",
    "surplus",
    "miscoding",
]]


## 9. Minimum improvement

The `min_improvement` argument controls how much the subset miscoding must improve before a new feature is accepted.

A larger value produces smaller selected subsets.


In [ ]:
selection_conservative = metric_redundant.select_features(
    min_improvement=0.05,
    return_details=True,
)

selection_conservative["selected_feature_names"]


In [ ]:
selection_conservative["path"]


## 10. Nonlinear example: comparison with correlation and mutual information

Correlation may miss nonlinear structure. Mutual information may detect dependence, but it does not separate deficiency from surplus.


In [ ]:
rng = np.random.default_rng(123)
n = 5000

x = rng.normal(size=n)
y_nonlinear = (x**2 > 1.0).astype(int)

X_nonlinear = pd.DataFrame({
    "nonlinear_signal": x,
    "linear_noise": rng.normal(size=n),
    "binary_noise": rng.integers(0, 2, size=n),
})

metric_nonlinear = Miscoding(
    X_type="auto",
    y_type="categorical",
    n_bins=20,
)
metric_nonlinear.fit(X_nonlinear, y_nonlinear)

show_feature_analysis(metric_nonlinear)


In [ ]:
comparison = pd.DataFrame(index=X_nonlinear.columns)

comparison["abs_correlation"] = [
    abs(np.corrcoef(X_nonlinear[col], y_nonlinear)[0, 1])
    for col in X_nonlinear.columns
]

comparison["mutual_information"] = mutual_info_classif(
    X_nonlinear,
    y_nonlinear,
    discrete_features=[False, False, True],
    random_state=42,
)

comparison["deficiency"] = metric_nonlinear.feature_deficiency()
comparison["surplus"] = metric_nonlinear.feature_surplus()
comparison["miscoding"] = metric_nonlinear.feature_miscoding()

comparison


In [ ]:
comparison[["abs_correlation", "mutual_information"]].plot(kind="bar")
plt.title("Nonlinear example: classical relevance scores")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.show()

comparison[["deficiency", "surplus", "miscoding"]].plot(kind="bar")
plt.title("Nonlinear example: nescience-oriented metrics")
plt.ylabel("Value")
plt.xticks(rotation=45)
plt.ylim(0, 1.05)
plt.show()


## 11. Redundant and noisy features in a classification dataset

This example uses a standard synthetic classification dataset with informative, redundant, and noisy features.

The goal is to show how the class analyzes feature-level miscoding, pairwise redundancy, subset miscoding, and greedy feature selection.


In [ ]:
X_cls, y_cls = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=3,
    n_redundant=3,
    n_repeated=0,
    n_classes=2,
    shuffle=False,
    random_state=42,
)

feature_names = (
    [f"informative_{i}" for i in range(3)]
    + [f"redundant_{i}" for i in range(3)]
    + [f"noise_{i}" for i in range(6)]
)

X_cls = pd.DataFrame(X_cls, columns=feature_names)

metric_cls = Miscoding(
    X_type="numeric",
    y_type="categorical",
    n_bins=20,
)
metric_cls.fit(X_cls, y_cls)

show_feature_analysis(metric_cls)


In [ ]:
plot_redundancy(
    metric_cls,
    "Classification example: pairwise feature redundancy",
).round(3)


In [ ]:
cls_selection = metric_cls.select_features(return_details=True)

cls_selection["selected_feature_names"]


In [ ]:
cls_selection["path"]


In [ ]:
cls_selection["subset"]


In [ ]:
cls_selection["features"][[
    "feature_name",
    "code_length",
    "deficiency",
    "surplus",
    "miscoding",
]]


## 12. Summary

The current interface focuses on feature-level and subset-level quantities:

Feature-level methods:

- `feature_deficiency()`;
- `feature_surplus()`;
- `feature_miscoding()`;
- `feature_analysis()`;
- `feature_redundancy()`.

Subset-level methods:

- `miscoding_subset(subset, mode="deficiency")`;
- `miscoding_subset(subset, mode="surplus")`;
- `miscoding_subset(subset, mode="miscoding")`;
- `subset_analysis(subset)`.

Selection method:

- `select_features(max_features=None, min_improvement=None, return_details=False)`.

The notebook avoids conditional-selection methods because the current class selects features by direct greedy reduction of subset-level miscoding.
